# Data Cleaning Pipeline

## Pipeline order

1. MA precincts (merge MGGG + 2024 election)
2. TX VTDs (merge MGGG + 2024 election)
3. Fix invalid TX geometries
4. Enrich TX with Asian population from Census PL2020
5. Congressional district GeoJSON (MA + TX)
6. Simplified web GeoJSON
7. State summary JSON
8. Gingles precinct scatter
9. Gingles regression curves
10. Congressional vote margins
11. Precinct adjacency
12. Enacted plan district demographics
13. Precinct heatmaps + census-block heatmaps + copy to client
14. Verification

Step 14 (block heatmaps) is heavy — TX alone reads ~8M census blocks and writes a ~260 MB GeoJSON.  Set `RUN_BLOCK_HEATMAPS = False` below to skip it on a fast iteration.

In [ ]:
import json
import os
import time
import warnings

warnings.filterwarnings("ignore")

from clean import (
    adjacency,
    block_heatmaps,
    client_outputs,
    congressional_districts,
    enacted_demographics,
    gingles,
    ma_precincts,
    state_summaries,
    tx_asian,
    tx_geometry,
    tx_vtds,
    verify,
    vote_margins,
    web_geojson,
)
from clean.deps import HAS_LIBPYSAL, HAS_LOWESS, HAS_SCIPY
from clean.paths import (
    ANALYSIS_DIR,
    CLIENT_DATA_DIR,
    GEOJSON_DIR,
    GERRYCHAIN_DIR,
    RAW_FILES,
    SUMMARY_DIR,
    ensure_dirs,
)

# Toggle the heavy block-heatmap step independently
RUN_BLOCK_HEATMAPS = True

ensure_dirs()

missing = [p for p in RAW_FILES.values() if not os.path.exists(p)]
assert not missing, f"Missing raw files: {missing}"
print(f"All {len(RAW_FILES)} raw files present.")
print(f"scipy:    {HAS_SCIPY}")
print(f"lowess:   {HAS_LOWESS}")
print(f"libpysal: {HAS_LIBPYSAL}")

## Step 1 — MA precincts

> **Prepro-1** *Integrate multiple data sources* · **Prepro-3** *Integrate enacted plan* (via `CONG_DIST`) · **Prepro-6** *Generate SeaWulf input files* (writes `cleaned/gerrychain/ma_precincts.shp`)

Joins MGGG geometry to the 2024 election shapefile on `(TOWN, WARD, PRECINCT)`.  Falls back to a centroid spatial join for rows that don't match by name.  Writes `cleaned/gerrychain/ma_precincts.shp`.

In [ ]:
ma = ma_precincts.run()
print(f"\nMA: {len(ma):,} precincts, {len(ma.columns)} columns")
ma.drop(columns="geometry")[["NAME", "TOWN", "VAP", "HVAP", "BVAP", "ASIANVAP",
                              "G24PREDHAR", "G24PRERTRU"]].head()

## Step 2 — TX VTDs

> **Prepro-1** *Integrate multiple data sources* · **Prepro-3** *Integrate enacted plan* (via `CONG_DIST`) · **Prepro-6** *Generate SeaWulf input files* (writes `cleaned/gerrychain/tx_vtds.shp`)

Joins MGGG geometry to two 2024 shapefiles — `all` (general election columns) and `cong` (`CONG_DIST`) — on `(COUNTYFP, VTD)`.  De-duplicates VTDs that get split across legislative districts and falls back to spatial join + `USCD` for unmatched rows.

In [ ]:
tx = tx_vtds.run()
print(f"\nTX: {len(tx):,} VTDs, {len(tx.columns)} columns")
tx.drop(columns="geometry")[["COUNTY", "CNTYVTD", "VAP", "HISPVAP", "BVAP",
                              "G24PREDHAR", "G24PRERTRU", "CONG_DIST"]].head()

## Step 3 — Fix invalid TX geometries

> Supports **Prepro-1** / **Prepro-6** — without valid polygons the GerryChain graph build would fail.

MGGG VTDs occasionally have self-intersections.  We run `buffer(0)` on any invalid polygon to repair it.  If anything was fixed, the shapefile is rewritten.

In [ ]:
tx = tx_geometry.run(tx)
still_invalid = (~tx.geometry.is_valid).sum()
print(f"\nRemaining invalid geometries: {still_invalid}")

## Step 4 — Enrich TX with Asian population

> **Prepro-1** *Integrate multiple data sources* — pulls Census PL2020 block-level Asian counts into the precinct table.

MGGG TX VTDs ship without `ASIAN` / `ASIANVAP` columns.  We aggregate Census PL2020 block-level NH-Asian counts (`P0020008`, `P0040008`) up to MGGG VTDs, allocating proportionally by `TOTPOP` for any MGGG VTDs that map to the same Census VTD (e.g. `0001A` and `0001B`).

In [ ]:
tx = tx_asian.run(tx)
print(f"\nTX ASIAN total:    {int(tx['ASIAN'].sum()):,}")
print(f"TX ASIANVAP total: {int(tx['ASIANVAP'].sum()):,}")
tx[["COUNTY", "CNTYVTD", "TOTPOP", "VAP", "ASIAN", "ASIANVAP"]].head()

## Step 5 — Congressional district GeoJSON

> **Prepro-1** / **Prepro-3** — converts existing district boundary data to a consistent GeoJSON format and integrates it with the dataset (powers GUI-2, GUI-6).

TIGER/Line `cd118` shapefiles → reproject to EPSG:4326 → simplify TX (MA is already small) → write `cleaned/geojson/{ma,tx}_congressional_districts.geojson`.

In [ ]:
congressional_districts.run()
for state in ["ma", "tx"]:
    p = os.path.join(GEOJSON_DIR, f"{state}_congressional_districts.geojson")
    print(f"  {state.upper()} CDs: {os.path.getsize(p)/1e6:.2f} MB at {p}")

## Step 6 — Simplified web GeoJSON

> Supports **Prepro-4** — produces lightweight client-facing artifacts of the preprocessed data.

Per-precinct/VTD data with simplified geometry (Douglas-Peucker) + reduced coordinate precision (~11 cm).  These feed the basic map layer in the client.

In [ ]:
web_geojson.run(ma, tx)
for fname in ["ma_precincts.geojson", "tx_vtds.geojson"]:
    p = os.path.join(GEOJSON_DIR, fname)
    print(f"  {fname}: {os.path.getsize(p)/1e6:.2f} MB")

## Step 7 — State summary JSON

> Supports **Prepro-4** — stores aggregate state metadata (powers GUI-3 *State data summary*).

Per-state aggregates: total population, VAP by group, presidential 2024 totals, congressional party split (from `congressional_reps.json`), and the list of *feasible* demographic groups (population > 400 k).

In [ ]:
state_summaries.run(ma, tx)
for state in ["ma", "tx"]:
    p = os.path.join(SUMMARY_DIR, f"{state}_state_summary.json")
    with open(p) as f:
        s = json.load(f)
    print(f"\n{s['state']} ({s['state_abbr']}):")
    print(f"  pop={s['total_population']:,}  vap={s['voting_age_population']:,}")
    print(f"  feasible groups: {s['feasible_demographic_groups']}")
    print(f"  pres D/R: {s['presidential_2024']['dem_pct']}% / {s['presidential_2024']['rep_pct']}%")

## Steps 8 & 9 — Gingles precinct + regression

> **Prepro-7** *Gingles 2/3 precinct analysis* (`gingles.run_precinct`) · **Prepro-8** *Gingles 2/3 non-linear regression* (`gingles.run_regression`)

For each (state, group): per-precinct minority-VAP % vs Democratic vote share, and a 200-point smoothed curve through it.  The cohesion test from *Thornburg v. Gingles* — see the dedicated `gingles_analysis.ipynb` for a deeper walk-through.

In [ ]:
gingles.run_precinct(ma, tx)
gingles.run_regression()

for state in ["ma", "tx"]:
    with open(os.path.join(ANALYSIS_DIR, f"{state}_gingles_precinct.json")) as f:
        scatter = json.load(f)
    with open(os.path.join(ANALYSIS_DIR, f"{state}_gingles_regression.json")) as f:
        curves = json.load(f)
    print(f"\n{state.upper()}:")
    for g in scatter:
        n_curve = len(curves.get(g, []))
        print(f"  {g:<10} {len(scatter[g]):>5} scatter points, {n_curve} curve points")

## Step 10 — Congressional vote margins

> Supports **GUI-6** *Display Congressional representation table* — not its own Prepro use case, but produced here as part of Prepro-4 storage.

For each district, sum 2024 presidential votes across its precincts/VTDs and compute `|D - R| / (D + R)`.  Margins are merged into `congressional_reps.json` next to each representative.

In [ ]:
vote_margins.run(ma, tx)
from clean.paths import REPS_FILE
with open(REPS_FILE) as f:
    reps = json.load(f)
# Show first 3 reps per state with their just-written margin
for sa in ["MA", "TX"]:
    print(f"\n{sa} reps (first 3):")
    for r in reps[sa]["representatives"][:3]:
        print(f"  D{r['district']:02d}: {r.get('name', '?'):<25} "
              f"D={r['dem_votes']:>7,}  R={r['rep_votes']:>7,}  "
              f"margin={r['vote_margin_pct']}%")

## Step 11 — Precinct adjacency

> **Prepro-2** *Identify precinct neighbors* — required (AD).

Per the spec: *"Identify two precincts as neighbors if they share a common boundary of at least 200 feet and the edges of each precinct are within 200 feet of its neighbors' edges."*

Builds `{idx: [neighbor_idx, …]}` for each state.  Uses `libpysal` Queen contiguity if installed, else falls back to a `shapely.STRtree` query with a CRS-aware buffer (61 m ≈ 200 ft for projected CRS, 0.00055° ≈ 61 m for geographic CRS).

TX adjacency takes a minute or two on the STRtree fallback.

In [ ]:
adjacency.run(ma, tx)
for state in ["ma", "tx"]:
    p = os.path.join(ANALYSIS_DIR, f"{state}_adjacency.json")
    with open(p) as f:
        adj = json.load(f)
    avg = sum(len(v) for v in adj.values()) / len(adj)
    isolated = sum(1 for v in adj.values() if not v)
    print(f"  {state.upper()}: {len(adj):,} nodes, avg neighbors={avg:.1f}, isolated={isolated}")

## Step 12 — Enacted plan district demographics

> **Prepro-3** *Integrate enacted plan with dataset* · **Prepro-11** *Box & whisker data for enacted plan* — produces the per-district minority-VAP overlay dots that appear on top of the ensemble box plots in GUI-17.

Aggregates VAP by group within each `CONG_DIST`.  This is what the GUI shows as the *enacted-plan baseline* the ensembles get compared against.

In [ ]:
enacted_demographics.run(ma, tx)
with open(os.path.join(ANALYSIS_DIR, "ma_enacted_demographics.json")) as f:
    ma_enacted = json.load(f)
print(f"MA: {len(ma_enacted['districts'])} districts")
for d in ma_enacted["districts"][:3]:
    pcts = {g: v["pct"] for g, v in d["groups"].items()}
    print(f"  D{d['district']:02d}: vap={d['total_vap']:,}  {pcts}")

## Client-facing heatmaps + analysis copy

> **Prepro-4** *Store preprocessed data* — the final fan-out into `client/public/data/`, where the React app reads everything without going through the API.

This is the work the original `main()` did *between* the analysis steps and verification:

- `client_outputs.write_precinct_heatmaps()` — slim per-precinct/VTD GeoJSON for the choropleth layer.
- `block_heatmaps.run()` — *step 14* in the old monolith.  Streams MA + TX census blocks straight to GeoJSON for the highest-zoom layer.
- `client_outputs.copy_analysis_to_client()` — copies summary + analysis JSON into `client/public/data/`.

In [ ]:
t0 = time.time()

client_outputs.write_precinct_heatmaps()

if RUN_BLOCK_HEATMAPS:
    block_heatmaps.run()
else:
    print("\n[skipped block_heatmaps — set RUN_BLOCK_HEATMAPS=True to enable]")

client_outputs.copy_analysis_to_client()
print(f"\nClient outputs done in {time.time()-t0:.1f}s")

## Step 13 — Verification

Reads everything we wrote and sanity-checks row counts, file sizes, presence of key columns, etc.  Returns `True` if all checks pass.

In [ ]:
ok = verify.run()
print(f"\nverify.run() -> {ok}")

## Final state

Everything written by this notebook lives under one of these directories.  `clean_data.py` would normally `rmtree` the `_tmp_unzip/` scratch — we leave it so re-running cells doesn't have to re-extract the raw zips.

In [ ]:
for d in [GERRYCHAIN_DIR, GEOJSON_DIR, SUMMARY_DIR, ANALYSIS_DIR, CLIENT_DATA_DIR]:
    files = sorted(os.listdir(d)) if os.path.exists(d) else []
    total_mb = sum(os.path.getsize(os.path.join(d, f)) for f in files) / 1e6
    print(f"\n{d} ({total_mb:.1f} MB total, {len(files)} files):")
    for f in files[:8]:
        sz = os.path.getsize(os.path.join(d, f)) / 1e6
        print(f"  {f:<45}  {sz:>8.2f} MB")
    if len(files) > 8:
        print(f"  ... +{len(files) - 8} more")